In [6]:
class Base:
    def __init__(self):
        self.pub = "base_pub"
        self._val = "base_val"

    @property
    def val(self):
        return self._val


class Impl(Base):
    def __init__(self):
        super().__init__()
        print(f"after super", self.__dict__)
        self._val = "impl_val"
        self.pub = "impl_pub"


data = Impl()

after super {'pub': 'base_pub', '_val': 'base_val'}


In [7]:
data.__dict__

{'pub': 'impl_pub', '_val': 'impl_val'}

In [8]:
data.val

'impl_val'

In [9]:
data.xxx = 123

In [10]:
data.__dict__

{'pub': 'impl_pub', '_val': 'impl_val', 'xxx': 123}

## Property

In [11]:
class Author:
    def __init__(self, name):
        self.name = name

    @property
    def name(self):
        print("Author.get_name")
        return f"{self.__name} {self.__surname}"

    @name.setter
    def name(self, val):
        print("Author.set_name", val)

        if not (isinstance(val, str) and len(val) >= 3 and " " in val):
            raise ValueError(f"wrong {val=} for name")

        self.__name, self.__surname = val.split()

    @name.deleter
    def name(self):
        print("Author.del_name")
        del self.__name


In [13]:
Author.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 1,
              '__init__': <function __main__.Author.__init__(self, name)>,
              'name': <property at 0x10bc735b0>,
              '__static_attributes__': ('__name', '__surname', 'name'),
              '__dict__': <attribute '__dict__' of 'Author' objects>,
              '__weakref__': <attribute '__weakref__' of 'Author' objects>,
              '__doc__': None})

In [14]:
steve = Author("Steve King")

Author.set_name Steve King


In [15]:
steve.__dict__

{'_Author__name': 'Steve', '_Author__surname': 'King'}

In [16]:
steve.name

Author.get_name


'Steve King'

In [17]:
steve.__dict__["name"] = "new_name"

In [18]:
steve.__dict__

{'_Author__name': 'Steve', '_Author__surname': 'King', 'name': 'new_name'}

In [19]:
steve.name

Author.get_name


'Steve King'

In [20]:
del steve.name

Author.del_name


In [21]:
steve.name

Author.get_name


AttributeError: 'Author' object has no attribute '_Author__name'

In [22]:
steve.name = 123

Author.set_name 123


ValueError: wrong val=123 for name

In [23]:
steve.name = "Steve KingNew"

Author.set_name Steve KingNew


In [24]:
steve.name

Author.get_name


'Steve KingNew'

In [25]:
def make_hash_from_password(password):
    return hash(password)


class User:
    def __init__(self, login, password):
        self.__login = login
        self.password_hash = None
        self.password = password

    @property
    def login(self):
        return self.__login

    @property
    def password(self):
        raise AttributeError("Password is write-only")

    @password.setter
    def password(self, plaintext):
        self.password_hash = make_hash_from_password(plaintext)

In [26]:
steve = User("steve", "12345")

In [27]:
steve.__dict__

{'_User__login': 'steve', 'password_hash': 7026008757576690253}

In [28]:
steve.login

'steve'

In [29]:
steve.login = "1234"

AttributeError: property 'login' of 'User' object has no setter

In [30]:
steve.login

'steve'

In [31]:
steve.password

AttributeError: Password is write-only

In [32]:
steve.password = "123456"

In [33]:
steve.__dict__

{'_User__login': 'steve', 'password_hash': -4591578656700735522}

## Дескрипторы

In [42]:
class MyDescriptor:
    def __get__(self, obj, objtype):
        print(f"get {obj} cls={objtype}")
    
    def __set__(self, obj, val):
        print(f"set {val} for {obj}")
    
    def __delete__(self, obj):
        print(f"delete from {obj}")


class MyClass:
    field = MyDescriptor()

In [43]:
inst = MyClass()

In [46]:
inst.field = 1

set 1 for <__main__.MyClass object at 0x10b727cb0>


In [35]:
MyClass.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 12,
              'field': <__main__.MyDescriptor at 0x10c1ac1a0>,
              '__static_attributes__': (),
              '__dict__': <attribute '__dict__' of 'MyClass' objects>,
              '__weakref__': <attribute '__weakref__' of 'MyClass' objects>,
              '__doc__': None})

In [36]:
MyClass.field

get None cls=<class '__main__.MyClass'>


In [39]:
MyClass.field = "123"

In [40]:
MyClass.field

'123'

In [41]:
MyClass.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 12,
              'field': '123',
              '__static_attributes__': (),
              '__dict__': <attribute '__dict__' of 'MyClass' objects>,
              '__weakref__': <attribute '__weakref__' of 'MyClass' objects>,
              '__doc__': None})

In [50]:
# BAD !!!


class PosIntField:
    def __init__(self):
        self.val = None

    def __get__(self, obj, objtype):
        print(f"get {obj} cls={objtype}")

        if obj is None:
            return

        return self.val

    def __set__(self, obj, val):
        print(f"set {val} for {obj}")

        if not (isinstance(val, int) and val >= 0):
            raise ValueError(f"wrong pos int {val=}")

        self.val = val

    def __delete__(self, obj):
        print(f"delete from {obj}")


class PersonModel:
    age = PosIntField()

    def __init__(self, age):
        self.age = age


steve = PersonModel(99)
tom = PersonModel(77)
print(f"{steve.age=}, {tom.age=}")

steve.age = 88

print("-" * 20)
print(f"{steve.age=}, {tom.age=}")

set 99 for <__main__.PersonModel object at 0x10c1ad2b0>
set 77 for <__main__.PersonModel object at 0x10c147c50>
get <__main__.PersonModel object at 0x10c1ad2b0> cls=<class '__main__.PersonModel'>
get <__main__.PersonModel object at 0x10c147c50> cls=<class '__main__.PersonModel'>
steve.age=77, tom.age=77
set 88 for <__main__.PersonModel object at 0x10c1ad2b0>
--------------------
get <__main__.PersonModel object at 0x10c1ad2b0> cls=<class '__main__.PersonModel'>
get <__main__.PersonModel object at 0x10c147c50> cls=<class '__main__.PersonModel'>
steve.age=88, tom.age=88


In [54]:
# BAD !!!


class PosIntField:
    def __init__(self):
        self.attr_name = "_pos_int_field"

    def __get__(self, obj, objtype):
        print(f"get {obj} cls={objtype}")

        if obj is None:
            return

        return getattr(obj, self.attr_name)

    def __set__(self, obj, val):
        print(f"set {val} for {obj}")

        if not (isinstance(val, int) and val >= 0):
            raise ValueError(f"wrong pos int {val=}")

        return setattr(obj, self.attr_name, val)

    def __delete__(self, obj):
        print(f"delete from {obj}")


class PersonModel:
    age = PosIntField()
    height = PosIntField()

    def __init__(self, age, height):
        self.age = age
        self.height = height


steve = PersonModel(99, 290)
tom = PersonModel(77, 222)
print("-" * 20)
print(f"{steve.age=}, {steve.height=}, {tom.age=}, {tom.height=}")

steve.age = 88
steve.height = 555

print("-" * 20)
print(f"{steve.age=}, {steve.height=}, {tom.age=}, {tom.height=}")

print("-" * 20)
print(f"{steve.__dict__=}, {tom.__dict__=}")

set 99 for <__main__.PersonModel object at 0x10c1ad7f0>
set 290 for <__main__.PersonModel object at 0x10c1ad7f0>
set 77 for <__main__.PersonModel object at 0x10c22ca50>
set 222 for <__main__.PersonModel object at 0x10c22ca50>
--------------------
get <__main__.PersonModel object at 0x10c1ad7f0> cls=<class '__main__.PersonModel'>
get <__main__.PersonModel object at 0x10c1ad7f0> cls=<class '__main__.PersonModel'>
get <__main__.PersonModel object at 0x10c22ca50> cls=<class '__main__.PersonModel'>
get <__main__.PersonModel object at 0x10c22ca50> cls=<class '__main__.PersonModel'>
steve.age=290, steve.height=290, tom.age=222, tom.height=222
set 88 for <__main__.PersonModel object at 0x10c1ad7f0>
set 555 for <__main__.PersonModel object at 0x10c1ad7f0>
--------------------
get <__main__.PersonModel object at 0x10c1ad7f0> cls=<class '__main__.PersonModel'>
get <__main__.PersonModel object at 0x10c1ad7f0> cls=<class '__main__.PersonModel'>
get <__main__.PersonModel object at 0x10c22ca50> cls=<

In [60]:
class PosIntField:
    def __init__(self):
        self.attr_name = None

    def __set_name__(self, owner, name):
        print(f"set_name {owner=} {name=}")
        
        self.attr_name = f"_pos_int_field_{name}"

    def __get__(self, obj, objtype):
        print(f"get {obj} cls={objtype}")

        if obj is None:
            return

        return getattr(obj, self.attr_name)

    def __set__(self, obj, val):
        print(f"set {val} for {obj}")

        if not (isinstance(val, int) and val >= 0):
            raise ValueError(f"wrong pos int {val=}")

        return setattr(obj, self.attr_name, val)

    def __delete__(self, obj):
        print(f"delete from {obj}")


class PersonModel:
    age = PosIntField()
    height = PosIntField()

    def __init__(self, age, height):
        self.age = age
        self.height = height

    def print(self):
        print(f"{self=}")


steve = PersonModel(99, 290)
tom = PersonModel(77, 222)
print("-" * 20)
print(f"{steve.age=}, {steve.height=}, {tom.age=}, {tom.height=}")

steve.age = 88
steve.height = 555

print("-" * 20)
print(f"{steve.age=}, {steve.height=}, {tom.age=}, {tom.height=}")

print("-" * 20)
print(f"{steve.__dict__=}, {tom.__dict__=}")

set_name owner=<class '__main__.PersonModel'> name='age'
set_name owner=<class '__main__.PersonModel'> name='height'
set 99 for <__main__.PersonModel object at 0x10c1ada90>
set 290 for <__main__.PersonModel object at 0x10c1ada90>
set 77 for <__main__.PersonModel object at 0x10c22c410>
set 222 for <__main__.PersonModel object at 0x10c22c410>
--------------------
get <__main__.PersonModel object at 0x10c1ada90> cls=<class '__main__.PersonModel'>
get <__main__.PersonModel object at 0x10c1ada90> cls=<class '__main__.PersonModel'>
get <__main__.PersonModel object at 0x10c22c410> cls=<class '__main__.PersonModel'>
get <__main__.PersonModel object at 0x10c22c410> cls=<class '__main__.PersonModel'>
steve.age=99, steve.height=290, tom.age=77, tom.height=222
set 88 for <__main__.PersonModel object at 0x10c1ada90>
set 555 for <__main__.PersonModel object at 0x10c1ada90>
--------------------
get <__main__.PersonModel object at 0x10c1ada90> cls=<class '__main__.PersonModel'>
get <__main__.PersonMod

In [61]:
steve.__dict__

{'_pos_int_field_age': 88, '_pos_int_field_height': 555}

In [62]:
steve.__class__.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 30,
              'age': <__main__.PosIntField at 0x10c1adfd0>,
              'height': <__main__.PosIntField at 0x10c22d950>,
              '__init__': <function __main__.PersonModel.__init__(self, age, height)>,
              'print': <function __main__.PersonModel.print(self)>,
              '__static_attributes__': ('age', 'height'),
              '__dict__': <attribute '__dict__' of 'PersonModel' objects>,
              '__weakref__': <attribute '__weakref__' of 'PersonModel' objects>,
              '__doc__': None})

In [63]:
steve.__dict__["age"] = "new_age"
steve.__dict__["print"] = "new_print"

In [64]:
steve.__dict__

{'_pos_int_field_age': 88,
 '_pos_int_field_height': 555,
 'age': 'new_age',
 'print': 'new_print'}

In [65]:
steve.age

get <__main__.PersonModel object at 0x10c1ada90> cls=<class '__main__.PersonModel'>


88

In [66]:
steve.print

'new_print'

In [67]:
del steve.print

In [68]:
steve.__dict__

{'_pos_int_field_age': 88, '_pos_int_field_height': 555, 'age': 'new_age'}

In [69]:
steve.print()

self=<__main__.PersonModel object at 0x10c1ada90>
